# Gene Intersection

- **Purpose:** Build the shared gene table that harmonizes the tumor and cell line feature spaces.
- **Inputs:** `data/raw/geneAnno.csv`, `data/raw/SCP542_UMIcount_data.txt`, `data/raw/GSE178341_crc10x_full_c295v4_submit.h5`.
- **Outputs:** `data/processed/genes_combined.csv` (written by `scripts/01_gene_intersection.py`; this notebook is the interactive version).
- **Result:** 28,617 genes shared by both datasets.


In [1]:
import pandas as pd
import scanpy as sc
import os

import h5py

In [2]:
DATA_DIR = os.path.join("..", "..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")

tumor_file = os.path.join(DATA_DIR, "raw", "GSE178341_crc10x_full_c295v4_submit.h5")
cell_line_file = os.path.join(DATA_DIR, "raw", "SCP542_UMIcount_data.txt")
gene_anno_file = os.path.join(DATA_DIR, "raw", "geneAnno.csv")

# Gene Annotation Mapping

In [3]:
df_gene_anno = pd.read_csv(gene_anno_file, usecols=["ensembl_gene_id", "external_gene_name"])
df_gene_anno["ensembl_gene_id_base"] = df_gene_anno["ensembl_gene_id"].str.split(".").str[0]
df_gene_anno.head(3)

,external_gene_name,ensembl_gene_id,ensembl_gene_id_base
0,DDX11L1,ENSG00000223972.5_2,ENSG00000223972
1,WASH7P,ENSG00000227232.5_2,ENSG00000227232
2,RP11-34P13.3,ENSG00000243485.5_4,ENSG00000243485


# Cell Line genes

In [4]:
df_cell_line = pd.read_csv(cell_line_file, sep="\t", usecols=[0], index_col=0, skiprows=2)
df_cell_line.index.name = "gene_symbol"

symbol_to_ensembl = df_gene_anno.set_index("external_gene_name")["ensembl_gene_id_base"].to_dict()
df_cell_line["gene_id"] = df_cell_line.index.map(
    lambda x: symbol_to_ensembl.get(x)
)
df_cell_line.dropna(inplace=True)

In [5]:
df_cell_line.head(3)

,gene_id
gene_symbol,
A1BG,ENSG00000121410
A1BG-AS1,ENSG00000268895
A1CF,ENSG00000148584


# Tumor genes

In [6]:
data = h5py.File(tumor_file, "r")
data["matrix"].keys()

<KeysViewHDF5 ['barcodes', 'data', 'features', 'indices', 'indptr', 'shape']>

In [7]:
data["matrix"]["features"].keys()

<KeysViewHDF5 ['_all_tag_keys', 'feature_type', 'genome', 'id', 'name']>

In [8]:
matrix_group = data["matrix"]
features = matrix_group["features"]

gene_ids = [x.decode() if isinstance(x, bytes) else x for x in features["id"][:]]
gene_names = [x.decode() if isinstance(x, bytes) else x for x in features["name"][:]]

df_tumor = pd.DataFrame(gene_ids, columns=["gene_id"])
df_tumor.index = gene_names
df_tumor.index.name = "gene_symbol"
df_tumor = df_tumor[df_tumor["gene_id"].str.startswith(("ENSG"))].copy()
df_tumor["gene_id_base"] = df_tumor["gene_id"].str.split(".").str[0]

df_tumor = df_tumor[~df_tumor["gene_id_base"].duplicated(keep="first")].copy()
df_tumor["gene_id"] = df_tumor["gene_id_base"].copy()
df_tumor.drop(columns=["gene_id_base"], inplace=True)

In [9]:
df_tumor.head(3)

,gene_id
gene_symbol,
RP11-34P13.3,ENSG00000243485
FAM138A,ENSG00000237613
OR4F5,ENSG00000186092


# Gene Intersection

In [10]:
df_combined = df_cell_line.merge(
    df_tumor,
    on="gene_id",
    how="inner"
)

ensembl_to_symbol = df_gene_anno.set_index("ensembl_gene_id_base")["external_gene_name"].to_dict()
df_combined["gene_symbol"] = df_combined["gene_id"].map(
    lambda x: ensembl_to_symbol.get(x)
)

In [11]:
df_combined.head(3)

,gene_id,gene_symbol
0,ENSG00000121410,A1BG
1,ENSG00000268895,A1BG-AS1
2,ENSG00000148584,A1CF


In [12]:
df_combined

,gene_id,gene_symbol
0,ENSG00000121410,A1BG
1,ENSG00000268895,A1BG-AS1
2,ENSG00000148584,A1CF
3,ENSG00000175899,A2M
4,ENSG00000166535,A2ML1
...,...,...
28612,ENSG00000182957,SPATA13
28613,ENSG00000214946,TBC1D26
28614,ENSG00000132286,TIMM10B
28615,ENSG00000155957,TMBIM4
